### Documents
Langchain implements a Document abstraction which is intendent to represent a unit of text and associated metadata . it has two components
- page content: a string representing a content.
- metadata : a dict containing arbitrary metadata . the metadata atribute can capture the information about sourse of the document, its relationship to other documents, and other information . Note that an individual document object often represent a chunk of a larger document.

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are the great companions, known for there loyalty and friendlyness",
        metadata = {"sourse":"mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata = {"sourse":"mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners , requiringly relative simple care.",
        metadata = {"sourse":"fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimiking human speech.",
        metadata = {"sourse":"bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need penty of space to hop around.",
        metadata = {"sourse":"mammal-pets-doc"},
    )
]

In [3]:
documents

[Document(metadata={'sourse': 'mammal-pets-doc'}, page_content='Dogs are the great companions, known for there loyalty and friendlyness'),
 Document(metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'sourse': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners , requiringly relative simple care.'),
 Document(metadata={'sourse': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimiking human speech.'),
 Document(metadata={'sourse': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need penty of space to hop around.')]

In [29]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key,model="llama-3.3-70b-versatile")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002558B5F5890>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002558B4FEA50>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [30]:
# pip install langchain_huggingface
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [31]:
# Vector Store
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents,embedding=embeddings)
vectorstore

In [32]:
vectorstore.similarity_search("cat")

[Document(id='5595051a-1c7a-4f47-975e-ec06ea614cc2', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='de78f8d8-c8e5-4eae-9034-cdc744e4833b', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='ad0c7ac0-c331-4524-be03-2053b2873857', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='a881f738-1180-492d-be12-fc14a62479eb', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]

In [33]:
vectorstore.similarity_search_with_relevance_scores("cat")

[(Document(id='5595051a-1c7a-4f47-975e-ec06ea614cc2', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.33878038939544586),
 (Document(id='de78f8d8-c8e5-4eae-9034-cdc744e4833b', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.33878038939544586),
 (Document(id='ad0c7ac0-c331-4524-be03-2053b2873857', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.33878038939544586),
 (Document(id='a881f738-1180-492d-be12-fc14a62479eb', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.33878038939544586)]

### Retrievers
Langchain Vectore store object do not subclass runnable , and cannot immediatly be integrated into langchain expression Language chain.
LangChain retrievers are runnable , so they implement a standard set of method and designed into incoperated in LCEL chains.

In [34]:
from typing import List
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)# it gives the top result after applying the similarity search
retriever.batch(["cat","dog"])

[[Document(id='a881f738-1180-492d-be12-fc14a62479eb', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='a48f1784-5329-4f6d-a8e5-86a90385254b', metadata={'sourse': 'mammal-pets-doc'}, page_content='Dogs are the great companions, known for there loyalty and friendlyness')]]

above method is not that much efficient so we go for another technique
Vectorestore implements an as_retriever method that will generate a retriever specially vectorestore retriever . These retreiver include specific search_type and search_kwargs attributes that identify what methods of the underlying vectore store to call and how to parameterise them  

In [35]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='a881f738-1180-492d-be12-fc14a62479eb', metadata={'sourse': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='a48f1784-5329-4f6d-a8e5-86a90385254b', metadata={'sourse': 'mammal-pets-doc'}, page_content='Dogs are the great companions, known for there loyalty and friendlyness')]]

In [36]:
# RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided the context only.
{question}
Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain = {"context":retriever,"question":RunnablePassthrough()}|prompt|llm
response = rag_chain.invoke("tell me about dog")
print(response.content)

According to the provided context, dogs are great companions, known for their loyalty and friendliness.
